In [1]:
include("../Envs/Env.jl")
include("../Algorithms/PPO-RNN.jl")

evaluate (generic function with 1 method)

In [3]:
using LinearAlgebra
BLAS.set_num_threads(24)

In [4]:
Threads.nthreads()




1

## 1. Prepare Environment

In [2]:
using RockSample

pomdp = RockSamplePOMDP(7, 8)
pomdp_name = "RS78"
bool_full_observability = false
env = Env(pomdp, bool_full_observability)
action_space = GetActionSpace(env)
function create_env()
    return Env(pomdp, bool_full_observability)
end

# define convert_o function
function POMDPs.convert_o(T::Type{<:AbstractArray}, o::Int64, m::RockSamplePOMDP)
    vec = zeros(Float32, 3)
    vec[o] = 1.0f0
    return vec
end

# define process action function
function process_action(action::Int, action_space::UnitRange{Int})
    len = length(action_space)
    idx = action - first(action_space) + 1
    (idx < 1 || idx > len) && error("Action $action not in action space")
    onehot = zeros(Float32, len)
    onehot[idx] = 1.0f0
    return onehot
end

process_action (generic function with 1 method)

## 2. Prepare Parameters

In [3]:
state_dim = GetObsDim(env)
# action_dim = length(action_space)
layer_size = 64
rnn_hidden_size = 64
gamma = discount(pomdp)
training_episodes = 10000
batch_size = 4096

RSState{8}([1, 1], Bool[1, 1, 1, 1, 0, 0, 0, 0])
Float32[1.0, 0.0, 0.0]


4096

## 3. Prepare PPO-RNN agent

In [10]:
# if want to use gpu, need to uncomment the below line, and use device=Flux.gpu
# using CUDA

# agent = PPORNNAgent(action_space, action_dim, state_dim;
#     hidden_dim=layer_size, 
#     rnn_hidden_size=rnn_hidden_size, 
#     batch_size=batch_size, 
#     device=Flux.cpu) 
agent = PPORNNAgent(action_space, state_dim;
    hidden_dim=layer_size, 
    rnn_hidden_size=rnn_hidden_size, 
    batch_size=batch_size, 
    device=Flux.cpu) 

PPORNNAgent(Chain(LSTM(16 => 64), Dense(64 => 64, tanh), Dense(64 => 13)), Chain(Dense(16 => 64, tanh), LSTM(64 => 64), Dense(64 => 64, tanh), Dense(64 => 1)), (layers = ((cell = (Wi = Leaf(Adam(eta=0.0003, beta=(0.9, 0.999), epsilon=1.0e-8), (Float32[0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0], Float32[0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0], (0.9, 0.999))), Wh = Leaf(Adam(eta=0.0003, beta=(0.9, 0.999), epsilon=1.0e-8), (Float32[0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0], Float32[0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0], (0.9, 0.999))), bias = Leaf(Adam(eta=0.0003, beta=(0.9, 0.999), epsilon=1.0e-8), (Float32[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], Float32[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], (0.9, 0.999)))

## 4. Train

In [9]:
# 训练
rewards, losses, evals = train!(create_env, agent, training_episodes)

CompositeException: TaskFailedException

    nested task error: Scalar indexing is disallowed.
    Invocation of getindex resulted in scalar indexing of a GPU array.
    This is typically caused by calling an iterating implementation of a method.
    Such implementations *do not* execute on the GPU, but very slowly on the CPU,
    and therefore should be avoided.
    
    If you want to allow scalar iteration, use `allowscalar` or `@allowscalar`
    to enable scalar iteration globally or for the operations in question.
    Stacktrace:
      [1] error(s::String)
        @ Base ./error.jl:35
      [2] errorscalar(op::String)
        @ GPUArraysCore ~/.julia/packages/GPUArraysCore/aNaXo/src/GPUArraysCore.jl:151
      [3] _assertscalar(op::String, behavior::GPUArraysCore.ScalarIndexing)
        @ GPUArraysCore ~/.julia/packages/GPUArraysCore/aNaXo/src/GPUArraysCore.jl:124
      [4] assertscalar(op::String)
        @ GPUArraysCore ~/.julia/packages/GPUArraysCore/aNaXo/src/GPUArraysCore.jl:1

In [ ]:
# 训练
rewards, losses, evals = train!(create_env, agent, training_episodes)

Progress:   1%|▍                                        |  ETA: 1 days, 4:47:39K

Update 100 | Threads: 1 | Reward: 5.11 | Policy Loss: -6.237236e10 | Value Loss: 0.9532 | Eval: 7.35


Progress:   2%|▉                                        |  ETA: 1 days, 4:07:12

Update 200 | Threads: 1 | Reward: 8.72 | Policy Loss: -6.237236e10 | Value Loss: 0.5983 | Eval: 7.35


Progress:   2%|▉                                        |  ETA: 1 days, 4:10:20

CompositeException: TaskFailedException

    nested task error: InterruptException:
    Stacktrace:
      [1] accum(x::Matrix{Float32}, ys::Matrix{Float32})
        @ Zygote ~/.julia/packages/Zygote/55SqB/src/lib/lib.jl:17
      [2] accum(x::Matrix{Float32}, y::ChainRulesCore.InplaceableThunk{ChainRulesCore.Thunk{ChainRules.var"#551#556"{Matrix{Float32}, Matrix{Float32}}}, ChainRules.var"#550#555"{Matrix{Float32}, Matrix{Float32}}})
        @ Zygote ~/.julia/packages/Zygote/55SqB/src/lib/lib.jl:50
      [3] macro expansion
        @ ~/.julia/packages/Zygote/55SqB/src/lib/lib.jl:21 [inlined]
      [4] accum
        @ ~/.julia/packages/Zygote/55SqB/src/lib/lib.jl:21 [inlined]
      [5] map
        @ ./tuple.jl:386 [inlined]
      [6] accum(x::Tuple{@NamedTuple{weight::Matrix{Float32}, bias::Vector{Float32}, σ::Nothing}, @NamedTuple{cell::@NamedTuple{Wi::Matrix{Float32}, Wh::Matrix{Float32}, bias::Vector{Float32}}}, @NamedTuple{weight::Matrix{Float32}, bias::Vector{Float32}, σ::Nothing}, 

In [ ]:
# 训练
rewards, losses, evals = train!(create_env, agent, training_episodes)

Progress:   1%|█                                        |  ETA: 1 days, 1:00:34m

Update 100 | Threads: 16 | Reward: 7.25 | Policy Loss: -0.0323 | Value Loss: 0.7434 | Eval: 7.35


Progress:   2%|█                                        |  ETA: 1 days, 1:17:54

Update 200 | Threads: 16 | Reward: 8.41 | Policy Loss: -0.0349 | Value Loss: 0.5887 | Eval: 7.35


Progress:   3%|██                                       |  ETA: 1 days, 0:53:41

...

Progress:  99%|█████████████████████████████████████████|  ETA: 0:28:30

Update 9900 | Threads: 16 | Reward: 14.57 | Policy Loss: -0.0477 | Value Loss: 0.3919 | Eval: 10.86


Progress: 100%|█████████████████████████████████████████| Time: 1 days, 23:06:28


Update 10000 | Threads: 16 | Reward: 14.37 | Policy Loss: -0.0491 | Value Loss: 0.3824 | Eval: 11.92


(Float32[-6.5625, -7.142857, -5.2777777, -6.571429, -4.0, 3.2432432, -3.7142856, -0.5405405, -8.888889, 2.368421  …  15.071428, 14.188235, 14.551887, 14.674699, 14.074074, 14.57346, 14.413145, 14.543325, 14.85646, 14.372093], Float32[-0.047519565, -0.04594249, -0.020245692, -0.032612614, -0.019958006, -0.03051366, -0.10142989, 0.004528658, -0.063719444, -0.011426655  …  -0.053054538, 0.22487752, -0.047674038, -0.049224194, -0.048624184, -0.05065903, -0.052065697, -0.05361969, 0.12140994, -0.04906578], Float32[1.0445323, 0.9413679, 1.0419542, 0.9353419, 0.9601696, 1.0388149, 0.8992502, 0.8815525, 0.92353565, 0.99426484  …  0.38140193, 0.4035726, 0.3907165, 0.38293687, 0.3928936, 0.3818751, 0.3841069, 0.3763733, 0.38605806, 0.38243604], Float32[7.350919, 7.350919, 7.350919, 9.099835, 9.075471, 9.71891, 9.321723, 10.239836, 8.392719, 11.569316  …  12.62761, 11.745698, 11.040169, 11.040169, 11.745698, 12.274846, 11.392934, 9.805491, 10.863787, 11.922081])

## 5. Evaluation

In [7]:
evaluate(env, agent; num_episodes=10000, max_steps=100) 

11.394697353317943

## (Todo) Save or plot the data from Train (rewards, losses, evals)